In [7]:
import bambi as bmb
import arviz as az

from utils import data

In [8]:
cluster_df = data.load_cluster_features()
cluster_df = cluster_df.to_pandas()

In [9]:
cluster_df.columns

Index(['window_id', 'window_idx', 'wn_mid_date', 'wn_prop_sequenced',
       'log_seq_prop', 'cluster_id', 'n_sequences', 'n_sequences_minus_one',
       'median_age', 'age_diversity', 'frac_female', 'frac_vaccinated',
       'simd_decile_mode', 'simd_decile_std', 'simd_quintile_mode',
       'simd_quintile_std', 'overall_zscore', 'income_zscore',
       'employment_zscore', 'education_zscore', 'health_zscore',
       'access_zscore', 'crime_zscore', 'housing_zscore', 'pango_lineage',
       'who_voc', 'qc_frac_mediocre', 'qc_frac_bad', 'wave'],
      dtype='object')

In [22]:
model_data = cluster_df.iloc[:, :].copy()

In [23]:
# Include all clusters, singletons (0) and non-singletons
formula = """
    n_sequences_minus_one ~
    offset(log_seq_prop) +
    median_age +
    age_diversity +
    frac_female +
    wave +
    C(simd_quintile_mode, Treatment(3)) +
    wave*C(simd_quintile_mode, Treatment(3)) +
    simd_quintile_std
"""

model = bmb.Model(
    formula,
    data=model_data,
    family="negativebinomial",
    dropna=True,
)

In [24]:
idata = model.fit(
    random_seed=42,
    idata_kwargs = {"log_likelihood": True},  # needed for model comparison
)
model.predict(idata, kind="response")  # "pps" = posterior predictive samples

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha, Intercept, median_age, age_diversity, frac_female, wave, C(simd_quintile_mode, Treatment(3)), wave:C(simd_quintile_mode, Treatment(3)), simd_quintile_std]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2015 seconds.


In [25]:
all_vars = list(idata.posterior.data_vars)

In [26]:
all_vars

['alpha',
 'Intercept',
 'median_age',
 'age_diversity',
 'frac_female',
 'wave',
 'C(simd_quintile_mode, Treatment(3))',
 'wave:C(simd_quintile_mode, Treatment(3))',
 'simd_quintile_std',
 'mu']

In [28]:
effect_vars = [
    'alpha',
     'Intercept',
     'median_age',
     'age_diversity',
     'frac_female',
     'wave',
     'C(simd_quintile_mode, Treatment(3))',
     'wave:C(simd_quintile_mode, Treatment(3))',
     'simd_quintile_std',
]
summary = az.summary(idata, var_names=effect_vars, hdi_prob=0.95, )
summary.index.name = "term"
summary.reset_index(inplace=True)
summary.to_csv("exploration_summary.csv", index=False)

In [30]:
idata.to_netcdf("exploration_summary.nc")

# idata = az.from_netcdf("exploration_summary.nc")

'exploration_summary.nc'